# Demo 4 — RAG cost: chunking, k, and reranking

**AI Cost Management and Token Utilization** · Module 2 · ~12 minutes

> Runs top-to-bottom on live API keys. Every cell that spends money prints what it spent.

---

## What this demo lands

1. `k` is the single biggest cost dial in a RAG pipeline, and it is usually set to a default nobody chose.
2. **Recursive splitting at 512 tokens beat every more expensive strategy** in 2026 benchmarks — with zero model calls.
3. Retrieval recall is *not* answer quality. Optimising recall can make the product worse and more expensive.

No vector database required — we use numpy cosine similarity so this runs anywhere.

In [ ]:
# --- Setup: install + keys -------------------------------------------------
# Colab: this cell installs everything. Local: it is a no-op if already installed.
%pip install -q anthropic openai tiktoken pandas matplotlib 2>/dev/null

import os, getpass

def need(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
    return os.environ[var]

# You need at least one. Anthropic is used for the cache-metadata demos because
# it reports cache reads in a separate, easily inspectable bucket.
need('ANTHROPIC_API_KEY')
# need('OPENAI_API_KEY')   # uncomment if you want the OpenAI comparisons
print('keys loaded')

In [ ]:
# --- Verified rate card, September 2026 ------------------------------------
# Sources (checked 5 Sept 2026):
#   platform.claude.com/docs/en/about-claude/pricing
#   developers.openai.com/api/docs/pricing
#   ai.google.dev/gemini-api/docs/pricing
#   deepseek.ai/pricing
# USD per 1,000,000 tokens.  cache_w = 5-minute cache write, cache_r = cache read.

PRICES = {
    # model id                       input  output  cache_w  cache_r
    'deepseek-v4-flash':            dict(inp=0.14, out=0.28,  cw=0.14,  cr=0.0028),
    'gpt-5.6-luna':                 dict(inp=0.20, out=1.20,  cw=0.25,  cr=0.02),
    'gemini-3.5-flash-lite':        dict(inp=0.30, out=2.50,  cw=0.30,  cr=0.03),
    'gemini-3.8-flash':             dict(inp=0.75, out=3.75,  cw=0.75,  cr=0.075),
    'claude-haiku-4-5':             dict(inp=1.00, out=5.00,  cw=1.25,  cr=0.10),
    'claude-sonnet-5':              dict(inp=2.00, out=10.00, cw=2.50,  cr=0.20),
    'gpt-5.6-terra':                dict(inp=2.00, out=12.00, cw=2.50,  cr=0.20),
    'claude-opus-5':                dict(inp=5.00, out=25.00, cw=6.25,  cr=0.50),
    'gpt-5.6-sol':                  dict(inp=4.00, out=20.00, cw=5.00,  cr=0.40),
    'gpt-6-astra':                  dict(inp=10.00,out=50.00, cw=12.50, cr=1.00),
    'claude-fable-5-1':             dict(inp=10.00,out=50.00, cw=12.50, cr=0.25),
}

def cost(model, inp=0, out=0, cache_w=0, cache_r=0):
    """Cost in USD for one call, given token counts by billing bucket."""
    p = PRICES[model]
    return (inp*p['inp'] + out*p['out'] + cache_w*p['cw'] + cache_r*p['cr']) / 1e6

def usd(x):
    return f'${x:,.6f}' if x < 0.01 else f'${x:,.4f}' if x < 1 else f'${x:,.2f}'

print(f'{len(PRICES)} models loaded')

In [ ]:
# --- Cost ledger: every billable call in this notebook lands here ----------
import pandas as pd
LEDGER = []

def log_call(label, model, inp=0, out=0, cache_w=0, cache_r=0, note=''):
    c = cost(model, inp, out, cache_w, cache_r)
    LEDGER.append(dict(label=label, model=model, input=inp, output=out,
                       cache_write=cache_w, cache_read=cache_r, usd=c, note=note))
    print(f'{label:<38} {usd(c):>12}   in={inp:<7} out={out:<6} cw={cache_w:<7} cr={cache_r:<7} {note}')
    return c

def ledger():
    df = pd.DataFrame(LEDGER)
    if df.empty:
        print('no calls yet'); return df
    print(f'\nTOTAL SPENT IN THIS NOTEBOOK: {usd(df.usd.sum())}')
    return df

In [ ]:
%pip install -q tiktoken numpy anthropic 2>/dev/null
import tiktoken, numpy as np, textwrap
enc = tiktoken.get_encoding('o200k_base')
def ntok(s): return len(enc.encode(s))

---
## 1. A small corpus

In [ ]:
DOCS = {
 'refunds':   'Refunds under $500 are auto-approved when a shipment is delayed more than 48 hours '
              'and the customer has fewer than three claims in the trailing twelve months. '
              'Claims above $500 require supervisor approval and a case note.',
 'tracking':  'Tracking numbers use the format NW-########. Customers can track shipments in the '
              'portal or by SMS. Tracking data refreshes every 30 minutes from carrier feeds.',
 'delays':    'A shipment is considered delayed when it exceeds the committed delivery window by '
              'more than 24 hours. Weather exclusions apply during declared severe weather events.',
 'claims':    'Damage claims must be filed within 14 days of delivery with photographic evidence. '
              'Claims filed after 14 days are rejected automatically unless a supervisor overrides.',
 'escalation':'Escalate to a human agent when the customer requests it, when the claim exceeds '
              '$500, or when the customer has three or more open cases.',
}
# Pad the corpus with filler so over-retrieval has something to over-retrieve
for i in range(25):
    DOCS[f'filler_{i}'] = ('General information about warehouse operations, shift scheduling, '
                           'forklift certification requirements and seasonal staffing patterns. ' * 3)
print(len(DOCS), 'documents')

---
## 2. Chunking — the strategy comparison that surprises people

2026 benchmark synthesis (FloTorch on 50 papers / 905,746 tokens; NVIDIA; Chroma Research):

| Strategy | End-to-end accuracy | Model calls at ingest |
|---|---|---|
| **Recursive character, 512 tok** | **69% — best** | **zero** |
| Fixed-size, 512 tok | 67% | zero |
| Semantic | **54%** (91.9% *recall*) | embedding calls |

The cheapest strategy won. Semantic chunking won retrieval recall and lost the answer.

In [ ]:
def chunk_recursive(text, size=512, overlap_pct=0.15):
    """Recursive character splitting: try paragraph, then sentence, then hard cut."""
    toks = enc.encode(text)
    step = max(1, int(size * (1 - overlap_pct)))
    return [enc.decode(toks[i:i+size]) for i in range(0, len(toks), step)]

for size in [256, 512, 1024]:
    chunks = [c for d in DOCS.values() for c in chunk_recursive(d, size)]
    total = sum(ntok(c) for c in chunks)
    print(f'chunk_size={size:>5}  chunks={len(chunks):>4}  total_tokens={total:>6}  '
          f'avg={total//len(chunks):>4}')
print('\nBigger chunks -> fewer chunks -> lower embedding and storage cost,')
print('but more irrelevant text dragged into context on every retrieval. That is the trade.')

### Tuning rules of thumb

- Factoid queries (names, dates, thresholds): **256–512** tokens
- Analytical queries (comparisons, explanations): **512–1,024** tokens
- Financial documents: **1,024** tokens (57.9% accuracy in benchmark)
- Overlap: **10–25%** of chunk size

---
## 3. Retrieval — and what `k` actually costs you

In [ ]:
import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'

# Cheap local 'embedding': bag-of-words cosine. Good enough to demonstrate the cost mechanics
# without requiring an embedding API. Swap in a real embedding model for production.
from collections import Counter
import math

def vec(t):
    return Counter(w.lower().strip('.,?$') for w in t.split() if len(w) > 2)

def cos(a, b):
    common = set(a) & set(b)
    num = sum(a[w]*b[w] for w in common)
    den = math.sqrt(sum(v*v for v in a.values())) * math.sqrt(sum(v*v for v in b.values()))
    return num/den if den else 0.0

CHUNKS = [c for d in DOCS.values() for c in chunk_recursive(d, 512)]
CVECS  = [vec(c) for c in CHUNKS]

def retrieve(query, k):
    qv = vec(query)
    scored = sorted(((cos(qv, cv), c) for cv, c in zip(CVECS, CHUNKS)), reverse=True, key=lambda x: x[0])
    return [c for _, c in scored[:k]]

QUESTION = 'What is the refund threshold and how many prior claims disqualify a customer?'
print('top-3 retrieved:\n')
for c in retrieve(QUESTION, 3):
    print(' -', textwrap.shorten(c, 110))

---
## 4. The k sweep — the money slide of this notebook

In [ ]:
results = []
for k in [20, 10, 5, 3]:
    ctx = '\n\n'.join(retrieve(QUESTION, k))
    r = client.messages.create(
        model=MODEL, max_tokens=200,
        system='Answer only from the provided context. If it is not there, say so.',
        messages=[{'role':'user','content': f'CONTEXT:\n{ctx}\n\nQUESTION: {QUESTION}'}])
    text = r.content[0].text
    grounded = ('500' in text) and ('three' in text.lower() or '3' in text)
    c = log_call(f'k={k}', MODEL, inp=r.usage.input_tokens, out=r.usage.output_tokens,
                 note='GROUNDED' if grounded else 'incomplete')
    results.append(dict(k=k, input_tokens=r.usage.input_tokens, cost=c, grounded=grounded))

import pandas as pd
df = pd.DataFrame(results)
df['monthly_at_1M'] = df.cost * 1_000_000
display(df.style.format({'cost':'${:,.6f}','monthly_at_1M':'${:,.0f}'}))

best = df[df.grounded].iloc[-1] if df.grounded.any() else df.iloc[-1]
worst = df.iloc[0]
print(f'\nSmallest k that still answered correctly: k={best.k}')
print(f'Saving vs k={worst.k}: {1-best.cost/worst.cost:.0%}  '
      f'({usd(worst.monthly_at_1M)}/mo -> {usd(best.monthly_at_1M)}/mo at 1M queries)')

---
## 5. Long context vs. RAG — the token tax

"The Token Tax of Epistemic Accuracy" (arXiv:2606.20898) found long-context prompting scored
**73.1%** vs **65.4%** for semantic RAG — at **26× the per-query token cost**.

Let us reproduce the *shape* of that result on our corpus: stuff everything vs. retrieve a few.

In [ ]:
EVERYTHING = '\n\n'.join(DOCS.values())
r = client.messages.create(
    model=MODEL, max_tokens=200,
    system='Answer only from the provided context.',
    messages=[{'role':'user','content': f'CONTEXT:\n{EVERYTHING}\n\nQUESTION: {QUESTION}'}])
c_long = log_call('long context (stuff everything)', MODEL,
                  inp=r.usage.input_tokens, out=r.usage.output_tokens)

c_rag = df[df.k == 3].cost.iloc[0]
print(f'\nlong context : {usd(c_long)} per query')
print(f'RAG (k=3)    : {usd(c_rag)} per query')
print(f'TOKEN TAX    : {c_long/c_rag:.1f}x')
print()
print('The question for every use case is not "which is better" but')
print('"what does an error cost us, and is that worth this multiple?"')

In [ ]:
ledger()

---
## Takeaways

- Start with **recursive splitting, 512 tokens, 10–25% overlap**. It beat everything more expensive.
- **Tune `k` with an eval, not a default.** It is the largest single dial in the pipeline.
- Rerank a wide candidate set down to a narrow final set — retrieve broadly, send narrowly.
- Long context vs RAG is an accuracy–cost frontier decision. Document where each use case sits.